# SW-15 — Le coup argumentatif : greffer AIF sur le coup ontologique (jumeau C#)

**Concept** : SW-14 a montré que le coup ontologique $\eta : L_t \to L_{t+1}$ s'incarne comme un **diff de graphe exécutable** (triplets ajoutés, verdict SHACL, delta owlrl, provenance par réification RDF + PROV). Ce carnet réalise la **greffe symétrique** : le graphe étendu n'est plus une ontologie de bibliothèque mais un **graphe d'argumentation AIF** — ici, le débat sur l'extension Ebook elle-même — et le coup étend le **vocabulaire argumentatif** : un nouveau schéma d'inférence et un nouveau type de conflit, plus les instances qui les habitent.

**Plan** — les quatre gestes de SW-14, exécutés sur un objet nouveau :

1. **Exercice 1** — le graphe de débat $L_t$ en triplets AIF, puis le coup $\eta$ (vocabulaire + instances) et son diff ;
2. **Exercice 2** — le juge SHACL : le coup conforme, et le **contrôle négatif** (un coup malformé rejeté) ;
3. **Exercice 3** — le delta d'inférences owlrl, puis la **bascule de l'extension grounded de Dung** avant/après coup ;
4. **Exercice 4** — la provenance du coup par réification RDF classique + PROV : quel agent a proposé le coup, et quand.

**Moteurs (politique du jumeau SW-14-CSharp)** — RDF, SPARQL et SHACL sont **natifs dotNetRDF** (`VDS.RDF`, `VDS.RDF.Shacl`) ; la fermeture OWL-RL passe par le **pont pythonnet** vers `owlrl`, dotNetRDF n'offrant pas de reasoner OWL-RL équivalent — le profil de chaque moteur est affiché à l'exécution. Le calcul de Dung (projection AF + étiquetage grounded) est un algorithme pur, porté nativement en C#.

**Sources** — vocabulaire AIF et taxonomie des schémas repris de l'ontologie Argumentum du dépôt (`ontologies/argumentum_fallacies.owl`, étudiée dans `Argumentation-Onto-01-AIF-OWL2-Python.ipynb`) ; sémantiques de Dung selon `Argumentation-03-Dung-AF-Semantics-Python.ipynb` (Dung 1995) ; pipeline (SHACL / OWL-RL / diff / provenance) selon `SW-14-Coup-Ontologique-Python.ipynb` et son jumeau C#. La syntaxe RDF-star (`<< s p o >>`, triples cités) est la cible conceptuelle documentée dans SW-10-RDFStar-Python ; la solution de l'exercice 4 teste la capacité réellement disponible dans le dotNetRDF chargé et exécute la réification classique.

In [1]:
#r "nuget: dotNetRDF, 3.5.2"
#r "nuget: pythonnet, 3.1.0"
// Vérification préalable des outils canoniques (règle F : moteurs réels, jamais de repli).
// dotNetRDF 3.5.2 : RDF, SPARQL et SHACL natifs ; pythonnet : pont vers owlrl côté Python.
using System;
using System.Linq;
using System.Collections.Generic;
using VDS.RDF;
using VDS.RDF.Parsing;
using VDS.RDF.Writing;
using VDS.RDF.Writing.Formatting;
using VDS.RDF.Query;
using VDS.RDF.Shacl;
using Python.Runtime;
// La ligne ci-dessous affiche la version réellement chargée — c'est sur elle (pas une version supposée)
// que les cellules suivantes s'exécutent.
Console.WriteLine($"dotNetRDF {typeof(IGraph).Assembly.GetName().Version} chargé ; RDF, SPARQL et SHACL natifs disponibles ; pont pythonnet prêt.");

Installed Packages dotNetRDF, 3.5.2 pythonnet, 3.1.0

dotNetRDF 3.5.2.0 chargé ; RDF, SPARQL et SHACL natifs disponibles ; pont pythonnet prêt.


### Lecture du résultat

La sortie affiche la version **dotNetRDF réellement chargée par le kernel** — `3.5.2` — c'est sur elle, et non sur une version supposée, que toutes les cellules suivantes s'exécutent. Le détail compte pour la suite du carnet : RDF, SPARQL et SHACL sont natifs dans dotNetRDF (`VDS.RDF.Shacl`), tandis que la fermeture OWL-RL de l'exercice 3 emprunte le pont pythonnet vers `owlrl` — la version de `owlrl` et de `rdflib` sera affichée à ce moment-là, à la source. C'est la même politique que le jumeau SW-14-CSharp : natif partout où le profil existe, pont explicite et affiché là où il n'existe pas.

## Contexte — le débat sur le coup ontologique

Le débat porte sur l'objet même de SW-14 : faut-il adopter la classe `Ebook` dans l'ontologie de la bibliothèque ? Le graphe $L_t$ ci-dessous modélise ce débat en **AIF** (Argument Interchange Format) :

- les **I-nodes** portent l'information (prémisses, conclusions, énoncés attaqués) ;
- les **RA-nodes** (*rule application*) sont les arguments proprement dits : prémisses $\vdash$ conclusion, chacun typé par un **schéma d'inférence** Walton matérialisé dans l'ontologie Argumentum (`ExpertOpinion_Inference`, `NegativeConsequences_Inference`, `Example_Inference`, `Bias_Inference`) ;
- les **CA-nodes** (*conflict application*) sont les attaques : qui attaque quoi.

Le pont vers le modèle de données AIF est explicite : l'ontologie Argumentum déclare taxonomiquement `aif:I-node`, `aif:RA-node`, `aif:CA-node`, `aif:Inference_Scheme`, `aif:Conflict` et les propriétés `arg:aifAttackedNode` / `aif:hasConflictedElement`. On reprend ces IRIs **tels quels** au niveau instance, avec deux propriétés de pont déclarées (`ax:from`, `ax:to`) pour les arêtes d'attaque.

## Exercice 1 — Le graphe de débat $L_t$ et le coup argumentatif

**Énoncé** : (a) construire le graphe AIF du débat ci-dessus ; (b) construire le coup $\eta$ proposé par la commission de révision — un **nouveau schéma d'inférence** `RepresentativeSample_Inference` (convention de nommage Argumentum `<Nom>_Inference`), un **nouveau type de conflit** `SelectionBiasRebuttal_Conflict` (`<Nom>_Conflict`), et les instances qui les utilisent ; (c) exhiber le diff de triplets $L_{t+1} \setminus L_t$.

In [2]:
// Exercice 1 — à compléter
// Étape 1 : déclarer le pont AIF dans gBase (classes + propriétés avec domaines/portées)
// Étape 2 : peupler les I-nodes du débat (rdfs:label en français)
// Étape 3 : construire RA1..RA4 (prémisses, conclusion, schéma) puis CA1..CA3
// Étape 4 : construire le coup dans gExt (schéma + conflit + instances) puis le diff
// Indice : miroir de la solution SW-14 — le diff est gExt.Triples sauf gBase.Triples
Graph gEssaiBase = null, gEssaiExt = null; // TODO étudiant : les deux graphes AIF
if (gEssaiBase == null) Console.WriteLine("Exercice à compléter : L_t, le coup et le diff de triplets");
else Console.WriteLine($"Graphes construits : base {gEssaiBase.Triples.Count}, extension {gEssaiExt.Triples.Count} triplets");

Exercice à compléter : L_t, le coup et le diff de triplets


### Solution — Exercice 1

In [3]:
// Solution complète — Exercice 1

// Préfixes partagés : chaque graphe du carnet les enregistre à la création
var PREFIXES = new (string, string)[] {
    ("aif", "http://www.arg.dundee.ac.uk/aif#"),
    ("arg", "https://www.argumentum.games/argumentum_fallacies.owl#"),
    ("ax",  "http://example.org/aif-ext#"),
    ("rdf", "http://www.w3.org/1999/02/22-rdf-syntax-ns#"),
    ("rdfs","http://www.w3.org/2000/01/rdf-schema#"),
    ("owl", "http://www.w3.org/2002/07/owl#"),
    ("xsd", "http://www.w3.org/2001/XMLSchema#"),
    ("sh",  "http://www.w3.org/ns/shacl#"),
    ("prov","http://www.w3.org/ns/prov#"),
};
Graph Nouveau() { var g = new Graph(); foreach (var (p, u) in PREFIXES) g.NamespaceMap.AddNamespace(p, new Uri(u)); return g; }
void Aj(IGraph g, string s, string p, string o) => g.Assert(g.CreateUriNode(s), g.CreateUriNode(p), g.CreateUriNode(o));

var gBase = Nouveau();
// --- Pont : modèle de nœuds AIF (classes réelles de l'ontologie Argumentum) ---
foreach (var c in new[] { "I-node", "RA-node", "CA-node", "Inference_Scheme", "Conflict" })
    Aj(gBase, $"aif:{c}", "rdf:type", "owl:Class");

// Propriétés du modèle de données : domaines/portées AIF + pont d'attaque
Aj(gBase, "aif:Premise", "rdf:type", "owl:ObjectProperty");
Aj(gBase, "aif:Premise", "rdfs:domain", "aif:RA-node");
Aj(gBase, "aif:Premise", "rdfs:range", "aif:I-node");
Aj(gBase, "aif:Conclusion", "rdf:type", "owl:ObjectProperty");
Aj(gBase, "aif:Conclusion", "rdfs:domain", "aif:RA-node");
Aj(gBase, "aif:Conclusion", "rdfs:range", "aif:I-node");
Aj(gBase, "aif:scheme", "rdf:type", "owl:ObjectProperty");
Aj(gBase, "aif:scheme", "rdfs:domain", "aif:RA-node");
Aj(gBase, "aif:scheme", "rdfs:range", "aif:Inference_Scheme");
foreach (var p in new[] { "ax:from", "ax:to" }) {
    Aj(gBase, p, "rdf:type", "owl:ObjectProperty");
    Aj(gBase, p, "rdfs:domain", "aif:CA-node");
}

// Schémas Walton matérialisés dans Argumentum (classes citées telles quelles)
foreach (var s in new[] { "ExpertOpinion_Inference", "NegativeConsequences_Inference",
                          "Example_Inference", "Bias_Inference" })
    Aj(gBase, $"aif:{s}", "rdf:type", "aif:Inference_Scheme");

// --- Les I-nodes du débat ---
var claims = new (string, string)[] {
    ("I1", "Les bibliothécaires recommandent l'accès numérique"),
    ("I2", "La maintenance numérique coûte cher"),
    ("I3", "Le budget d'extension est serré"),
    ("I4", "Le pilote papier-vers-numérique affiche 82% de satisfaction"),
    ("I5", "Le pilote n'a interrogé que des membres déjà numériques"),
    ("I6", "La satisfaction du pilote n'est pas représentative"),
    ("C1", "Adopter la classe Ebook dans l'ontologie"),
    ("C2", "Reporter l'extension de l'ontologie"),
};
foreach (var (nid, txt) in claims) {
    Aj(gBase, $"ax:{nid}", "rdf:type", "aif:I-node");
    gBase.Assert(gBase.CreateUriNode($"ax:{nid}"), gBase.CreateUriNode("rdfs:label"), gBase.CreateLiteralNode(txt, "fr"));
}

// --- Les RA-nodes (arguments) : prémisses, conclusion, schéma ---
void Ra(IGraph g, string nid, string schema, string[] premisses, string conclusion) {
    Aj(g, $"ax:{nid}", "rdf:type", "aif:RA-node");
    Aj(g, $"ax:{nid}", "aif:scheme", schema);
    foreach (var p in premisses) Aj(g, $"ax:{nid}", "aif:Premise", $"ax:{p}");
    Aj(g, $"ax:{nid}", "aif:Conclusion", $"ax:{conclusion}");
}
Ra(gBase, "RA1", "aif:ExpertOpinion_Inference",        new[] { "I1" }, "C1");
Ra(gBase, "RA2", "aif:NegativeConsequences_Inference", new[] { "I2", "I3" }, "C2");
Ra(gBase, "RA3", "aif:Example_Inference",              new[] { "I4" }, "C1");
Ra(gBase, "RA4", "aif:Bias_Inference",                 new[] { "I5" }, "I6");

// --- Les CA-nodes (attaques) : qui attaque quoi ---
void Ca(IGraph g, string nid, string attaquant, string cible) {
    Aj(g, $"ax:{nid}", "rdf:type", "aif:CA-node");
    Aj(g, $"ax:{nid}", "ax:from", $"ax:{attaquant}");
    Aj(g, $"ax:{nid}", "arg:aifAttackedNode", $"ax:{cible}");
}
Ca(gBase, "CA1", "RA2", "RA1");   // undercut : l'argument de coût frappe le lien expertise -> adoption
Ca(gBase, "CA2", "RA1", "RA2");   // rebuttal : l'analyse des experts couvre déjà les coûts
Ca(gBase, "CA3", "RA4", "I4");    // undermine : le biais de sélection atteint la prémisse du pilote

Console.WriteLine($"État initial L_t : {gBase.Triples.Count} triplets");
Console.WriteLine("  I-nodes : 8    RA-nodes : 4 (RA1..RA4)    CA-nodes : 3 (CA1..CA3)");

// --- Le coup eta : L_t -> L_{t+1} ---
var gExt = Nouveau();
gExt.Merge(gBase);

// 1. Vocabulaire : nouveau schéma + nouveau type de conflit
Aj(gExt, "ax:RepresentativeSample_Inference", "rdf:type", "aif:Inference_Scheme");
Aj(gExt, "ax:RepresentativeSample_Inference", "rdfs:subClassOf", "aif:Inference_Scheme");
Aj(gExt, "ax:SelectionBiasRebuttal_Conflict", "rdf:type", "aif:Conflict");
Aj(gExt, "ax:SelectionBiasRebuttal_Conflict", "rdfs:subClassOf", "aif:Conflict");

// 2. Instances : preuve corrigeant le biais, argument qui la porte, attaque qui retourne RA4
Aj(gExt, "ax:I7", "rdf:type", "aif:I-node");
gExt.Assert(gExt.CreateUriNode("ax:I7"), gExt.CreateUriNode("rdfs:label"),
            gExt.CreateLiteralNode("L'étude complémentaire couvre tous les profils de membres", "fr"));
Aj(gExt, "ax:I8", "rdf:type", "aif:I-node");
gExt.Assert(gExt.CreateUriNode("ax:I8"), gExt.CreateUriNode("rdfs:label"),
            gExt.CreateLiteralNode("Le biais du pilote est corrigé", "fr"));
// idiome Argumentum : le schéma est matérialisé comme classe, l'argument en est instance
Ra(gExt, "RA5", "ax:RepresentativeSample_Inference", new[] { "I7" }, "I8");   // schéma du coup
Aj(gExt, "ax:RA5", "rdf:type", "ax:RepresentativeSample_Inference");

Aj(gExt, "ax:CA4", "rdf:type", "aif:CA-node");
Aj(gExt, "ax:CA4", "rdf:type", "ax:SelectionBiasRebuttal_Conflict");   // type de conflit du coup
Aj(gExt, "ax:CA4", "ax:from", "ax:RA5");
Aj(gExt, "ax:CA4", "arg:aifAttackedNode", "ax:I5");                    // undermine : la prémisse du biais tombe

// --- Le diff de triplets ---
var fmtAif = new TurtleFormatter(gBase);
var diffList = gExt.Triples.Except(gBase.Triples).OrderBy(t => t.ToString()).ToList();
Console.WriteLine($"\nLe coup ajoute {diffList.Count} triplets :");
Console.WriteLine(new string('-', 60));
foreach (var t in diffList)
    Console.WriteLine($"  {fmtAif.Format(t.Subject)} {fmtAif.Format(t.Predicate)} {fmtAif.Format(t.Object)} .");

État initial L_t : 64 triplets


  I-nodes : 8    RA-nodes : 4 (RA1..RA4)    CA-nodes : 3 (CA1..CA3)



Le coup ajoute 17 triplets :


------------------------------------------------------------


  ax:CA4 ax:from ax:RA5 .


  ax:CA4 rdf:type ax:SelectionBiasRebuttal_Conflict .


... (11 outputs left out)

  ax:RepresentativeSample_Inference rdf:type aif:Inference_Scheme .


  ax:RepresentativeSample_Inference rdfs:subClassOf aif:Inference_Scheme .


  ax:SelectionBiasRebuttal_Conflict rdf:type aif:Conflict .


  ax:SelectionBiasRebuttal_Conflict rdfs:subClassOf aif:Conflict .


### Interprétation — le coup comme double extension

Le diff se lit en deux étages, exactement comme le coup ontologique de SW-14 :

- **vocabulaire** (4 triplets) : `ax:RepresentativeSample_Inference` et `ax:SelectionBiasRebuttal_Conflict` sont déclarés `rdfs:subClassOf` des classes AIF existantes — le coup n'invente pas un formalisme, il **étend la taxonomie Argumentum** en suivant ses conventions de nommage ;
- **instances** (13 triplets) : deux I-nodes, un RA-node typé par le schéma du coup, un CA-node typé par le conflit du coup. L'attaque `CA4` vise `I5` — la *prémisse* de l'argument de biais `RA4` — et non sa conclusion : c'est un **undermine**, symétrique du `CA3` initial.

Un détail technique porte tout le reste : les arêtes d'attaque utilisent `arg:aifAttackedNode` (l'IRI de l'ontologie Argumentum) avec `ax:from` pour l'attaquant — le pont minimal entre la taxonomie Argumentum et le modèle de nœuds AIF.

## Exercice 2 — L'admissibilité SHACL et le contrôle négatif

**Énoncé** : (a) écrire les shapes qui encodent les **contraintes structurelles AIF** (un RA-node a au moins une prémisse, exactement une conclusion, un schéma ; un CA-node a un attaquant typé et une cible ; un I-node porte un libellé) ; (b) valider le coup (cas A, attendu conforme) ; (c) construire et valider un **coup malformé** (cas B, attendu rejeté) — un argument sans prémisse ni schéma, une attaque sans attaquant.

In [4]:
// Exercice 2 — à compléter
// Étape 1 : écrire les 3 NodeShapes (RAShape, CAShape, IShape) avec leurs property shapes
// Étape 2 : valider gExt (cas A, attendu conforme)
// Étape 3 : construire gBad — un RA-node sans prémisse ni schéma, un CA-node sans attaquant — et valider
// Indice : miroir de la solution SW-14 — sh:minCount, sh:maxCount, sh:class
Graph gEssaiShapes = null; // TODO étudiant : les shapes SHACL du modèle AIF
if (gEssaiShapes == null) Console.WriteLine("Exercice à compléter : le juge SHACL et son contrôle négatif");
else Console.WriteLine($"Shapes : {gEssaiShapes.Triples.Count} triplets");

Exercice à compléter : le juge SHACL et son contrôle négatif


### Solution — Exercice 2

In [5]:
// Solution complète — Exercice 2

// Les shapes SHACL : contraintes structurelles du modèle AIF
// (6 property shapes : RAShape x3, CAShape x2, IShape x1)
var gShapes = Nouveau();
var ttlShapes = @"ax:RAShape a sh:NodeShape ; sh:targetClass aif:RA-node ;
  sh:property [ sh:path aif:Premise ; sh:minCount 1 ] ;
  sh:property [ sh:path aif:Conclusion ; sh:minCount 1 ; sh:maxCount 1 ] ;
  sh:property [ sh:path aif:scheme ; sh:minCount 1 ; sh:maxCount 1 ] .
ax:CAShape a sh:NodeShape ; sh:targetClass aif:CA-node ;
  sh:property [ sh:path ax:from ; sh:minCount 1 ; sh:class aif:RA-node ] ;
  sh:property [ sh:path arg:aifAttackedNode ; sh:minCount 1 ] .
ax:IShape a sh:NodeShape ; sh:targetClass aif:I-node ;
  sh:property [ sh:path rdfs:label ; sh:minCount 1 ] .
";
foreach (var (p, u) in PREFIXES) gShapes.NamespaceMap.AddNamespace(p, new Uri(u));
var entetePrefixes = string.Join("\n", PREFIXES.Select(p => $"@prefix {p.Item1}: <{p.Item2}> .")) + "\n";
StringParser.Parse(gShapes, entetePrefixes + ttlShapes);   // l'en-tête @prefix oriente la détection vers Turtle
Console.WriteLine($"Shapes chargés : {gShapes.Triples.Count} triplets (3 NodeShapes, 6 property shapes)");

var shapes = new ShapesGraph(gShapes);
void Verdict(string nom, IGraph donnees) {
    var report = shapes.Validate(donnees);
    Console.WriteLine(new string('=', 60));
    Console.WriteLine(nom);
    Console.WriteLine(new string('=', 60));
    Console.WriteLine($"  conforms  : {report.Conforms}");
}

// --- CAS A : le coup conforme ---
Verdict("CAS A — le coup de la commission (gExt)", gExt);

// --- CAS B : le coup malformé (contrôle négatif) ---
var gBad = Nouveau(); gBad.Merge(gBase);
// un argument aveugle : conclusion sans prémisse ni schéma
Aj(gBad, "ax:RA6", "rdf:type", "aif:RA-node");
Aj(gBad, "ax:RA6", "aif:Conclusion", "ax:C2");
// une attaque orpheline : cible sans attaquant
Aj(gBad, "ax:CA5", "rdf:type", "aif:CA-node");
Aj(gBad, "ax:CA5", "arg:aifAttackedNode", "ax:I1");

var reportB = shapes.Validate(gBad);
Console.WriteLine();
Verdict("CAS B — le coup malformé (RA6 sans prémisse ni schéma, CA5 sans attaquant)", gBad);

if (!reportB.Conforms) {
    Console.WriteLine("  - détail des violations :");
    int nViol = 0;
    foreach (var r in reportB.Results.OrderBy(r => r.FocusNode.ToString())) {
        nViol++;
        var chemin = r.ResultPath;
        Console.WriteLine($"    violation #{nViol}: focusNode={fmtAif.Format(r.FocusNode)}, " +
                          $"path={(chemin == null ? "(absente)" : fmtAif.Format(chemin))}");
    }
}

Shapes chargés : 27 triplets (3 NodeShapes, 6 property shapes)


CAS A — le coup de la commission (gExt)


  conforms  : True


CAS B — le coup malformé (RA6 sans prémisse ni schéma, CA5 sans attaquant)


  conforms  : False


  - détail des violations :


    violation #1: focusNode=ax:CA5, path=ax:from


    violation #2: focusNode=ax:RA6, path=aif:Premise


    violation #3: focusNode=ax:RA6, path=aif:scheme


### Interprétation — un juge qui a dit non

Le cas A passe : le coup de la commission respecte toutes les contraintes structurelles du modèle AIF. Le cas B, lui, est **rejeté** sur (au moins) trois violations : `RA6` n'a ni prémisse ni schéma (un argument qui n'en est pas un — une affirmation nue), et `CA5` n'a pas d'attaquant (une attaque venue de nulle part).

C'est ce couple conforme/rejeté qui donne sa valeur au verdict SHACL : un juge qui n'aurait jamais dit non ne discriminerait rien. La grille AIF encode ce qu'un **coup argumentatif admissible** doit être — et le cas B montre qu'elle mord.

## Exercice 3 — Le delta d'inférences et la bascule de Dung

**Énoncé** : (a) mesurer le delta d'inférences owlrl $\mathrm{cl}(L_{t+1}) \setminus \mathrm{cl}(L_t)$ ; (b) projeter le graphe AIF en cadre d'argumentation abstrait (AF de Dung) — un argument est un RA-node, une attaque défaite l'argument visé directement (undercut) ou celui dont la prémisse est visée (undermine) ; (c) calculer l'extension **grounded** avant et après le coup, et exhiber la bascule.

In [6]:
// Exercice 3 — à compléter
// Étape 1 : fermeture owlrl de gBase et gExt, puis le delta d'inférences
// Étape 2 : ProjeterAF — arguments = RA-nodes, attaques = CA-nodes (undercut + undermine par prémisse)
// Étape 3 : étiquetage grounded avant/après le coup
// Indice : un argument non attaqué entre IN ; un argument est défendu quand tous ses attaquants sont OUT
List<string> basculeEssai = null; // TODO étudiant : la bascule de l'extension grounded
if (basculeEssai == null) Console.WriteLine("Exercice à compléter : delta owlrl et bascule de l'extension grounded");
else Console.WriteLine($"Bascule : {string.Join(", ", basculeEssai)}");

Exercice à compléter : delta owlrl et bascule de l'extension grounded


### Solution — Exercice 3

In [7]:
// Solution complète — Exercice 3

// --- (a) Delta d'inférences owlrl : pont pythonnet (même politique que SW-14-CSharp) ---
if (!PythonEngine.IsInitialized) {
    Runtime.PythonDLL = Environment.GetEnvironmentVariable("PYTHONNET_PYDLL")
        ?? throw new Exception("Définir PYTHONNET_PYDLL avant le démarrage du kernel.");
    PythonEngine.Initialize();
    PythonEngine.BeginAllowThreads();
}
using (Py.GIL()) {
    dynamic owl = Py.Import("owlrl"); dynamic rdf = Py.Import("rdflib");
    Console.WriteLine($"Moteur owlrl {owl.__version__} ; rdflib {rdf.__version__}");
}
HashSet<string> Fermer(IGraph data) {
    var nt = VDS.RDF.Writing.StringWriter.Write(data, new NTriplesWriter());
    using (Py.GIL()) {
        dynamic rdf = Py.Import("rdflib"); dynamic owl = Py.Import("owlrl");
        dynamic graph = rdf.Graph(); graph.parse(data: nt, format: "nt");
        owl.DeductiveClosure(owl.OWLRL_Semantics).expand(graph);
        string result = graph.serialize(format: "nt").ToString();
        return result.Split('\n', StringSplitOptions.RemoveEmptyEntries).ToHashSet();
    }
}

var infBase = Fermer(gBase);
var infExt  = Fermer(gExt);
var deltaInf = infExt.Except(infBase).OrderBy(s => s).ToList();
Console.WriteLine($"cl(L_t)   : {infBase.Count} triplets");
Console.WriteLine($"cl(L_t+1) : {infExt.Count} triplets");
Console.WriteLine($"\nDelta d'inférences : {deltaInf.Count} nouveaux triplets");
Console.WriteLine(new string('-', 60));
Graph Mini(IEnumerable<string> lignes) {
    var g = new Graph(); StringParser.Parse(g, string.Join("\n", lignes), new NTriplesParser()); return g;
}
// Les triplets à sujet littéral (sameAs owlrl entre littéraux identiques) ne sont pas
// sérialisables en NTriples standard — même filtre que SW-14-CSharp pour la relecture.
var deltaAffichable = deltaInf.Where(l => !l.StartsWith("\"")).ToList();
foreach (var t in Mini(deltaAffichable.Take(12)).Triples.OrderBy(t => t.ToString()))
    Console.WriteLine($"  {fmtAif.Format(t.Subject)} {fmtAif.Format(t.Predicate)} {fmtAif.Format(t.Object)} .");
if (deltaInf.Count > 12) Console.WriteLine($"  ... et {deltaInf.Count - 12} de plus.");

// --- (b) Projection AIF -> AF abstrait de Dung (natif C#) ---
string Frag(INode n) { var s = n.ToString(); return s.Contains('#') ? s.Split('#').Last() : s; }
List<string> Fmt(IEnumerable<INode> ext) => ext.Select(Frag).OrderBy(x => x).ToList();

HashSet<INode> ArgsDe(IGraph g) {
    var t = g.GetTriplesWithPredicateObject(g.CreateUriNode("rdf:type"), g.CreateUriNode("aif:RA-node"));
    return t.Select(x => x.Subject).ToHashSet();
}
Dictionary<INode, HashSet<INode>> PremissesDe(IGraph g) {
    var d = new Dictionary<INode, HashSet<INode>>();
    foreach (var tr in g.GetTriplesWithPredicate(g.CreateUriNode("aif:Premise"))) {
        if (!d.TryGetValue(tr.Object, out var s)) { s = new HashSet<INode>(); d[tr.Object] = s; }
        s.Add(tr.Subject);
    }
    return d;
}
HashSet<(INode a, INode b)> AttaquesDe(IGraph g) {
    var args = ArgsDe(g); var prem = PremissesDe(g);
    var att = new HashSet<(INode, INode)>();
    foreach (var tr in g.GetTriplesWithPredicateObject(g.CreateUriNode("rdf:type"), g.CreateUriNode("aif:CA-node"))) {
        var ca = tr.Subject;
        var f = g.GetTriplesWithSubjectPredicate(ca, g.CreateUriNode("ax:from")).FirstOrDefault()?.Object;
        var c = g.GetTriplesWithSubjectPredicate(ca, g.CreateUriNode("arg:aifAttackedNode")).FirstOrDefault()?.Object;
        if (f == null || c == null) continue;
        if (args.Contains(c)) att.Add((f, c));                    // undercut : l'attaque vise un argument
        if (prem.TryGetValue(c, out var owners))                 // undermine : la prémisse tombe, l'argument tombe
            foreach (var o in owners) if (!o.Equals(f)) att.Add((f, o));
    }
    return att;
}

// --- (c) Étiquetage IN/OUT/UNDEC jusqu'au point fixe (sémantique grounded, Dung 1995) ---
HashSet<INode> Grounded(HashSet<INode> args, HashSet<(INode a, INode b)> attaques) {
    var attaquants = args.ToDictionary(a => a, a => attaques.Where(t => t.b.Equals(a)).Select(t => t.a).ToHashSet());
    var label = args.ToDictionary(a => a, _ => "UNDEC");
    bool change = true;
    while (change) {
        change = false;
        foreach (var a in args) {
            if (label[a] != "UNDEC") continue;
            if (attaquants[a].Any(b => label[b] == "IN")) { label[a] = "OUT"; change = true; }
            else if (attaquants[a].All(b => label[b] == "OUT")) { label[a] = "IN"; change = true; }
        }
    }
    return args.Where(a => label[a] == "IN").ToHashSet();
}

var argsB = ArgsDe(gBase);  var attB = AttaquesDe(gBase);
var G_avant = Grounded(argsB, attB);
var argsE = ArgsDe(gExt);   var attE = AttaquesDe(gExt);
var G_apres = Grounded(argsE, attE);

Console.WriteLine("\n" + new string('=', 60));
Console.WriteLine("Projection AF et extension grounded");
Console.WriteLine(new string('=', 60));
Console.WriteLine($"AVANT le coup : AF = {argsB.Count} arguments, {attB.Count} attaques");
Console.WriteLine($"  attaques       : {string.Join(", ", attB.Select(t => Frag(t.a) + " -> " + Frag(t.b)).OrderBy(x => x))}");
Console.WriteLine($"  grounded       : [{string.Join(", ", Fmt(G_avant))}]");
Console.WriteLine($"APRÈS le coup  : AF = {argsE.Count} arguments, {attE.Count} attaques");
Console.WriteLine($"  attaques       : {string.Join(", ", attE.Select(t => Frag(t.a) + " -> " + Frag(t.b)).OrderBy(x => x))}");
Console.WriteLine($"  grounded       : [{string.Join(", ", Fmt(G_apres))}]");

var gagnes = G_apres.Except(G_avant); var perdus = G_avant.Except(G_apres);
Console.WriteLine($"\nBascule du coup : +[{string.Join(", ", Fmt(gagnes))}] GAGNÉS, -[{string.Join(", ", Fmt(perdus))}] PERDUS");

Moteur owlrl 7.1.4 ; rdflib 7.6.0


cl(L_t)   : 270 triplets


cl(L_t+1) : 305 triplets



Delta d'inférences : 35 nouveaux triplets


------------------------------------------------------------


  ax:CA4 ax:from ax:RA5 .


... (16 outputs left out)

APRÈS le coup  : AF = 5 arguments, 4 attaques


  attaques       : RA1 -> RA2, RA2 -> RA1, RA4 -> RA3, RA5 -> RA4


  grounded       : [RA3, RA5]



Bascule du coup : +[RA3, RA5] GAGNÉS, -[RA4] PERDUS


### Interprétation — une conséquence qui se mesure deux fois

**Côté owlrl**, le delta n'est pas vide : la re-typification par les nouvelles sous-classes propage (`rdfs9`) — `RA5` devient instance de `Inference_Scheme`, `CA4` instance de `Conflict` — et les domaines `aif:scheme` / `ax:from` re-typent les nouveaux nœuds. Le coup a des conséquences **déductives**.

**Côté Dung**, la conséquence est plus forte : le coup **bascule l'extension grounded**. Avant, `RA4` (l'argument de biais, inattaquable) était `IN` et tenait `RA3` (l'argument du pilote) en `OUT` — le débat sur l'adoption restait flottant (`RA1`/`RA2` mutuellement attaquants, `UNDEC`). Après, `RA5` — inattaquable — fait tomber la prémisse `I5` de `RA4` : `RA4` passe `OUT`, `RA3` est **réhabilité** `IN`. L'extension grounded passe de `{RA4}` à `{RA3, RA5}`.

C'est la différence entre une **décoration** (des triplets en plus sans effet) et un **coup** : ici, une étude complémentaire typée par un nouveau schéma renverse le verdict d'acceptabilité d'un argument existant. La même distinction que le coup nul de SW-14, mais vue du côté de la controverse.

## Exercice 4 — La provenance du coup (réification RDF classique + PROV)

**Énoncé** : (a) encapsuler le triplet signature du coup (`RA5 rdf:type ax:RepresentativeSample_Inference`) par **réification RDF classique** (`rdf:Statement`) et lui attacher la provenance : quel agent a proposé le coup, quand ; (b) interrogation SPARQL : retrouver les éléments du coup attribués à cet agent.

> **Note de transparence** : la syntaxe RDF-star (`<< s p o >>`, triples cités) est la **cible** conceptuelle — présentée comme telle dans SW-10-RDFStar-Python. La cellule de solution teste la capacité réellement offerte par le dotNetRDF chargé (détection du type triple cité) et exécute la **réification classique**, l'implémentation pratique et interrogeable par SPARQL 1.1. C'est la double lecture établie par SW-10.

In [8]:
// Exercice 4 — à compléter
// Étape 1 : encapsuler le triplet signature du coup en rdf:Statement
// Étape 2 : attacher prov:wasAttributedTo (l'agent) et prov:generatedAtTime
// Étape 3 : requête SPARQL listant les coups attribués à cet agent
// Indice : miroir de la solution SW-14 — l'encapsulation puis le filtre sur l'agent
Graph gEssaiProv = null; // TODO étudiant : le graphe avec provenance
if (gEssaiProv == null) Console.WriteLine("Exercice à compléter : la provenance du coup (réification + PROV)");
else Console.WriteLine($"Provenance : {gEssaiProv.Triples.Count} triplets");

Exercice à compléter : la provenance du coup (réification + PROV)


### Solution — Exercice 4

In [9]:
// Solution complète — Exercice 4

// Capacité RDF-star (triple cité) ? Même convention que SW-10 : on teste la CAPACITÉ,
// pas le numéro de version. La réification classique rdf:Statement est l'implémentation
// pratique ; RDF-star (<< s p o >>) reste la syntaxe cible documentée.
var typeTripleCite = typeof(IGraph).Assembly.GetTypes().FirstOrDefault(t => t.Name == "TripleNode");
Console.WriteLine($"Support RDF-star natif (triple cité) : {(typeTripleCite != null ? "OUI" : "NON — réification classique utilisée")}");

var gProv = Nouveau(); gProv.Merge(gExt);

// --- Réification RDF classique du triplet signature (rdf:Statement) ---
var assertion = gProv.CreateBlankNode();
gProv.Assert(assertion, gProv.CreateUriNode("rdf:type"), gProv.CreateUriNode("rdf:Statement"));
gProv.Assert(assertion, gProv.CreateUriNode("rdf:subject"), gProv.CreateUriNode("ax:RA5"));
gProv.Assert(assertion, gProv.CreateUriNode("rdf:predicate"), gProv.CreateUriNode("rdf:type"));
gProv.Assert(assertion, gProv.CreateUriNode("rdf:object"), gProv.CreateUriNode("ax:RepresentativeSample_Inference"));

// --- Provenance : QUI a proposé le coup, et QUAND ---
gProv.Assert(gProv.CreateUriNode("ax:agentCommission"), gProv.CreateUriNode("rdfs:label"),
             gProv.CreateLiteralNode("Commission de révision des débats", "fr"));
gProv.Assert(assertion, gProv.CreateUriNode("prov:wasAttributedTo"), gProv.CreateUriNode("ax:agentCommission"));
gProv.Assert(assertion, gProv.CreateUriNode("prov:generatedAtTime"),
             gProv.CreateLiteralNode("2026-08-30T12:00:00", new Uri("http://www.w3.org/2001/XMLSchema#dateTime")));

Console.WriteLine($"Graphe avec provenance : {gProv.Triples.Count} triplets");

// --- Interrogation SPARQL ---
var sparqlQ = @"PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX ax:   <http://example.org/aif-ext#>

SELECT ?agent ?sujet ?objet ?moment
WHERE {
  ?assertion a rdf:Statement ;
             rdf:subject ?sujet ;
             rdf:predicate rdf:type ;
             rdf:object ?objet ;
             prov:wasAttributedTo ?agent ;
             prov:generatedAtTime ?moment .
  FILTER(?agent = ax:agentCommission)
}";
var resultats = (SparqlResultSet)gProv.ExecuteQuery(sparqlQ);
Console.WriteLine($"\nRésultat SPARQL : {resultats.Count} coup(s) attribué(s)");
foreach (SparqlResult row in resultats) {
    Console.WriteLine($"  agent   = {fmtAif.Format(row["agent"])}");
    Console.WriteLine($"  triplet = {fmtAif.Format(row["sujet"])} rdf:type {fmtAif.Format(row["objet"])}");
    Console.WriteLine($"  moment  = {fmtAif.Format(row["moment"])}");
}

Support RDF-star natif (triple cité) : OUI


Graphe avec provenance : 88 triplets



Résultat SPARQL : 1 coup(s) attribué(s)


  agent   = ax:agentCommission


  triplet = ax:RA5 rdf:type ax:RepresentativeSample_Inference


  moment  = "2026-08-30T12:00:00"^^xsd:dateTime


### Interprétation — la provenance comme mémoire du geste

La réification RDF classique du triplet signature + `prov:wasAttributedTo` nomme **l'auteur du coup** : la commission de révision, à l'instant où elle l'a proposé. Comme dans SW-14, cette trace répond à la question qui surgit dès qu'un vocabulaire est étendu sous controverse : *qui a proposé ce schéma, et au nom de quoi ?* La requête SPARQL la rend interrogeable mécaniquement — la mémoire du débat est dans le graphe, pas dans un paratexte.

RDF-star (`<< s p o >>`) ferait la même chose avec une syntaxe plus compacte (le triplet signé comme sujet de l'annotation, sans recours à `rdf:Statement`) — la cellule de solution exécute cette détection — sur le dotNetRDF chargé ici, elle rend OUI (le nœud triple cité existe) — et la réification classique reste l'implémentation exécutée : interrogeable par SPARQL 1.1 standard, elle est le dénominateur commun des deux jumeaux.

## Conclusion — la greffe complète

Les quatre gestes du coup ontologique de SW-14, réexécutés sur un graphe d'argumentation AIF :

| Geste | SW-14 (ontologie bibliothèque) | SW-15 (graphe AIF) |
|---|---|---|
| Extension $\eta$ | classe `Ebook` + propriété `fmt` | schéma `RepresentativeSample_Inference` + conflit `SelectionBiasRebuttal_Conflict` + instances |
| Juge SHACL | `fmt` obligatoire sur `Ebook` | contraintes structurelles AIF ; coup malformé rejeté (contrôle négatif) |
| Conséquence | delta owlrl non vide | delta owlrl **et** bascule de l'extension grounded `{RA4} -> {RA3, RA5}` |
| Provenance | `ex:alice` attribue l'assertion Ebook (réification + PROV) | `ax:agentCommission` attribue le coup argumentatif (réification + PROV) |

La greffe tient en une phrase : **le coup ontologique est le format naturel d'un coup argumentatif** — étendre le vocabulaire d'un débat (nouveaux schémas, nouveaux types de conflit) est une opération sur $L_t$ dont l'admissibilité se juge (SHACL) et la conséquence se mesure (owlrl + Dung). Ce que SW-14 montrait sur une ontologie se transplante sans changement d'outils sur la controverse elle-même — côté C#, avec les mêmes moteurs que le jumeau SW-14-CSharp : dotNetRDF natif pour RDF/SPARQL/SHACL, pont pythonnet pour la fermeture owlrl, calcul de Dung natif.

**Perspective** — le prolongement naturel est le lien avec les notebooks `Argument_Analysis_*` : la taxonomie complète des schémas Walton (1 509 classes) comme réserve de vocabulaire pour de futurs coups, et les sémantiques preferred/stable comme juges alternatifs au grounded choisi ici (le plus sceptique — donc le plus difficile à faire basculer : un coup qui déplace le grounded déplace a fortiori les autres).